# Problem Statement
**Problem statement:** Can we forecast the number of deaths due to covid for three months with a 90% predicition accuracy?

**Background:** COVID-19 is an emergent disease that had no historical data to guide predicting/forecasting over time.

**Context/situation:** The ability to predict the progress of the disease is crucial to decision making aimed at controlling the spread

**Objectives:** To forecast the number of deaths due to covid-19 with a 90% prediction accuracy (p-values and uncertainty come into play here, will adjust my objective once I know more the importance of these statistical metrics with regards to time series modeling)

**Models (potentially):**
Autoregressive integrated moving average (ARIMA)
Autoregressive Poision model (ARPOIS)
cubic smoothing spline
NAIVE
Holt model – linear trend method, and winter seasonal method
Trigonometric Exponential smoothing state space model with Box-Cox transformation (TBATS)

**Metrics:** these are metrics for ARIMA, still need to look into others
ME
MAE
MSE
MPE (mean percentage error)
MAPE (mean absolute percentage error)
MASE (mean absolute scaled error)


In [1]:
# Imports
import pandas as pd
import numpy as np
import missingno as msno
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split

Importing One Month of Data

In [1]:
jhu = pd.read_csv('../datasets/csse_covid_19_daily_reports_us/06-01-2020.csv',
                 na_values=[''], keep_default_na=False)

NameError: name 'pd' is not defined

In [17]:
jhu.shape

(58, 18)

In [18]:
jhu.head()

,Province_State,Country_Region,Last_Update,Lat,Long_,Confirmed,Deaths,Recovered,Active,FIPS,Incident_Rate,People_Tested,People_Hospitalized,Mortality_Rate,UID,ISO3,Testing_Rate,Hospitalization_Rate
0,Alabama,US,2020-06-02 02:33:16,32.3182,-86.9023,18525,646,9355.0,8629.0,1,379.957110,223523.0,1856.0,3.467525,84000001,USA,4558.730703,9.962426
1,Alaska,US,2020-06-02 02:33:16,61.3707,-152.4044,466,10,368.0,88.0,2,63.700798,54190.0,NaN,2.145923,84000002,USA,7407.609921,NaN
2,American Samoa,US,2020-06-02 02:33:16,-14.2710,-170.1320,0,0,NaN,0.0,60,0.000000,174.0,NaN,NaN,16,ASM,312.719038,NaN
3,Arizona,US,2020-06-02 02:33:16,33.7298,-111.4312,20129,918,4869.0,14342.0,4,276.545990,228070.0,3018.0,4.560584,84000004,USA,3133.381886,14.993293
4,Arkansas,US,2020-06-02 02:33:16,34.9697,-92.3731,7443,133,5401.0,1909.0,5,246.636296,133236.0,711.0,1.786914,84000005,USA,4414.998456,9.552600


In [19]:
# Define a function that converts csv files from June-December 2020 into data frames and returns a data frame
def csv_concatenator(my_num):
    if my_num < 10:
        for i in range(6, 10):
            df = pd.read_csv(f'../datasets/csse_covid_19_daily_reports_us/0{i}-0{my_num}-2020.csv',
                            na_values=[''], keep_default_na=False)
    elif my_num >= 10:
        for i in range(6, 10):
            df = pd.read_csv(f'../datasets/csse_covid_19_daily_reports_us/0{i}-{my_num}-2020.csv',
                            na_values=[''], keep_default_na=False)  
    elif my_num < 10:
        for i in range(10, 13):
            df = pd.read_csv(f'../datasets/csse_covid_19_daily_reports_us/{i}-0{my_num}-2020.csv',
                            na_values=[''], keep_default_na=False)
    elif my_num >= 10:
        for i in range(10, 13):
            df = pd.read_csv(f'../datasets/csse_covid_19_daily_reports_us/{i}-{my_num}-2020.csv',
                            na_values=[''], keep_default_na=False)  
    return df

In [20]:
# Create a for loop to concatenate all data frames into 1 data frame
counter = jhu.shape[0]
for i in range(1, 32):
    try:
        jhu_i = csv_concatenator(i)
        counter += jhu_i.shape[0]
        jhu = pd.concat([jhu, jhu_i])
    except OSError as e:
        print(e.errno)
print(counter)

2
1798


In [21]:
jhu.shape

(1798, 18)

In [23]:
jhu.tail()

,Province_State,Country_Region,Last_Update,Lat,Long_,Confirmed,Deaths,Recovered,Active,FIPS,Incident_Rate,People_Tested,People_Hospitalized,Mortality_Rate,UID,ISO3,Testing_Rate,Hospitalization_Rate
53,Virginia,US,2020-10-01 04:30:28,37.7693,-78.1700,148271,3208,17633.0,127254.0,51.0,1735.008732,2049988.0,NaN,2.164195,84000051,USA,24017.145296,NaN
54,Washington,US,2020-10-01 04:30:28,47.4009,-121.4905,89463,2100,NaN,85396.0,53.0,1149.352985,1854399.0,NaN,2.429104,84000053,USA,24352.266013,NaN
55,West Virginia,US,2020-10-01 04:30:28,38.4912,-80.9545,15850,355,11507.0,3988.0,54.0,884.414058,561568.0,NaN,2.239748,84000054,USA,31334.929557,NaN
56,Wisconsin,US,2020-10-01 04:30:28,44.2685,-89.6165,122274,1327,99925.0,21022.0,55.0,2100.049567,1552370.0,NaN,1.085268,84000055,USA,26661.873711,NaN
57,Wyoming,US,2020-10-01 04:30:28,42.7560,-107.3025,5948,50,4791.0,1107.0,56.0,1027.716200,101160.0,NaN,0.840619,84000056,USA,17478.777868,NaN


In [15]:
# this was pulling it over 10k rows earlier...now it's just pulling one month worth of data. will have to fix this

In [24]:
# Rename column names
jhu.columns = [column.lower() for column in jhu.columns]

### Missing Values

In [25]:
jhu.isnull().sum()

province_state             0
country_region             0
last_update                0
lat                       62
long_                     62
confirmed                  0
deaths                     0
recovered                313
active                     0
fips                       0
incident_rate             62
people_tested             62
people_hospitalized     1763
mortality_rate            31
uid                        0
iso3                       0
testing_rate              62
hospitalization_rate    1763
dtype: int64

In [27]:
# Check columns with 62 missing values each
missing_df = jhu.loc[jhu['lat'].isnull()][['lat', 'long_', 
                                           'incident_rate', 
                                           'testing_rate']]
missing_df

,lat,long_,incident_rate,testing_rate
9,NaN,NaN,NaN,NaN
13,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN
13,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN
...,...,...,...,...
13,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN
13,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN


In [28]:
# Confirming that all these missing values came from the same indices
missing_df.isnull().sum()

lat              62
long_            62
incident_rate    62
testing_rate     62
dtype: int64

In [29]:
# Check column 'recovered'
jhu[['recovered']].head(3)

,recovered
0,9355.0
1,368.0
2,NaN


**Interpretation of Missing Values:** 

It appears that the 62 missing values for 'lat', 'long_', 'incident_rate', 'total_test_results', and 'testing_rate' all came from the same source. These data can be classified as missing at random. The columns for 'recovered' and 'case_fatality_ratio' are also missing values, but the pattern is not as obvious.

In the section on imputation, we will use the KNNImputer to fill in gaps with the missing values as we prefer this method over the complete case analysis, which would significantly reduce the amount of data with which we can work. Utilizing the KNN imputation method makes sense for the 5 columns with 62 missing values, since they all come from the same source. The type of missingness is not as clear for 'recovered' and 'case_fatality_ratio', so we assume that the KNNImputer will impute the best estimated values for the missing data in these two cases.